# Week 2 — Real Brain Data: The Sensorium Mouse

Last week we learned Python and numpy on *pretend* data, and discovered that a (simulated) neuron fired more when the mouse ran. **This week we use REAL recorded data** from a mouse's visual cortex.

This dataset comes from the **Sensorium 2022 competition**. Scientists showed a mouse thousands of natural images while recording:
- the **activity of 8,372 neurons** in its primary visual cortex (area V1),
- and, at the same time, the mouse's **behavior**: pupil size, how fast its pupil was changing, and its running speed.

**Our goal today:** load this real data, look around, and ask our big question for the first time on real neurons — *does the mouse's behavior relate to how its brain responds?*

> Runs in Google Colab. It downloads ~424 MB, so use a session with a decent connection. No GPU needed this week.

## 1. Download the data

The data lives on a scientific data repository (gin.g-node.org). We download one mouse and unzip it. This cell takes a couple of minutes — run it and wait for the ✅.

In [ ]:
FILENAME = "static21067-10-18-GrayImageNet-94c6ff995dac583098847cfecd43e7b6"
URL = f"https://gin.g-node.org/cajal/Sensorium2022/raw/master/{FILENAME}.zip"

import os
if not os.path.exists(FILENAME):
    print("Downloading (~424 MB)...")
    !wget -q --show-progress -O mouse.zip "$URL"
    print("Unzipping...")
    !unzip -q -o mouse.zip
    print("Done.")
else:
    print("Already downloaded.")

print("✅ Ready.")

## 2. What's inside?

The data is organized into folders. Let's look at the structure instead of assuming it.

In [ ]:
DATA_DIR = os.path.join(FILENAME, "data")

print("Folders inside 'data':", os.listdir(DATA_DIR))
for folder in ["images", "responses", "behavior", "pupil_center"]:
    n_files = len(os.listdir(os.path.join(DATA_DIR, folder)))
    print(f"  {folder:14s}: {n_files} files")

Each **trial** (one image shown to the mouse) has a matching file in every folder. So `responses/0.npy`, `images/0.npy`, and `behavior/0.npy` all describe **the same moment**.

| Folder | What one file contains |
|--------|------------------------|
| `images/` | the grayscale image shown to the mouse (144 × 256 pixels) |
| `responses/` | how much each of the 8,372 neurons fired |
| `behavior/` | 3 numbers: **[pupil size, change in pupil size, running speed]** |
| `pupil_center/` | where the eye was pointing (x, y) |


## 3. Look at a single trial

Let's load trial number 0 and see everything the mouse experienced and did at that moment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

trial = 0
image    = np.load(f"{DATA_DIR}/images/{trial}.npy")
response = np.load(f"{DATA_DIR}/responses/{trial}.npy")
behavior = np.load(f"{DATA_DIR}/behavior/{trial}.npy").ravel()

print("image shape:   ", image.shape, "  (channels, height, width)")
print("response shape:", response.shape, " (one number per neuron)")
print("behavior:      ", behavior, " = [pupil size, change in pupil, running speed]")

plt.figure(figsize=(6,3.5))
plt.imshow(image[0], cmap="gray")
plt.title("The image the mouse saw on trial 0")
plt.axis("off")
plt.show()

## 4. Load ALL trials into two big tables

To study patterns we want *every* trial together. We'll build:
- `R` — a table of neural responses, one **row per trial**, one **column per neuron**.
- `B` — a table of behavior, one row per trial, 3 columns.

This loop reads ~6,000 files, so it takes a few seconds.

In [ ]:
n_trials  = len(os.listdir(f"{DATA_DIR}/responses"))
n_neurons = np.load(f"{DATA_DIR}/responses/0.npy").shape[0]
print(f"Loading {n_trials} trials x {n_neurons} neurons...")

R = np.zeros((n_trials, n_neurons), dtype=np.float32)
B = np.zeros((n_trials, 3), dtype=np.float64)
for i in range(n_trials):
    R[i] = np.load(f"{DATA_DIR}/responses/{i}.npy")
    B[i] = np.load(f"{DATA_DIR}/behavior/{i}.npy").ravel()
    if i % 1500 == 0:
        print(f"  ...{i}")

# Pull out the behavior variables into clearly-named arrays
pupil_size    = B[:, 0]
pupil_change  = B[:, 1]
running_speed = B[:, 2]

print("Done!")
print("R (responses) table shape:", R.shape)
print("B (behavior)  table shape:", B.shape)

## 5. Get to know the behavior

Before comparing behavior to the brain, let's understand the behavior itself. **Always look at your data first** — it often surprises you.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(pupil_size, bins=40)
axes[0].set_title("Pupil size across trials")
axes[0].set_xlabel("pupil size"); axes[0].set_ylabel("# of trials")
axes[1].hist(running_speed, bins=40)
axes[1].set_title("Running speed across trials")
axes[1].set_xlabel("running speed"); axes[1].set_ylabel("# of trials")
plt.tight_layout(); plt.show()

frac_running = np.mean(running_speed > 0) * 100
print(f"The mouse was running (speed > 0) on only {frac_running:.1f}% of trials.")

**Notice something important:** this mouse was mostly sitting still — its running speed is zero on the large majority of trials. Its pupil, on the other hand, changed a lot. Keep this in mind: *we can only detect an effect of running if the mouse actually runs.*

## 6. One neuron: does its firing relate to behavior?

Just like Week 1's scatter plot — but with a real neuron. We'll compare one neuron's response to pupil size and to running speed.

In [ ]:
neuron = 0   # try changing this number later!

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(pupil_size, R[:, neuron], alpha=0.2, s=8)
axes[0].set_xlabel("pupil size"); axes[0].set_ylabel(f"neuron {neuron} response")
c_pupil = np.corrcoef(pupil_size, R[:, neuron])[0, 1]
axes[0].set_title(f"vs pupil   (correlation = {c_pupil:.2f})")

axes[1].scatter(running_speed, R[:, neuron], alpha=0.2, s=8)
axes[1].set_xlabel("running speed"); axes[1].set_ylabel(f"neuron {neuron} response")
c_run = np.corrcoef(running_speed, R[:, neuron])[0, 1]
axes[1].set_title(f"vs running   (correlation = {c_run:.2f})")
plt.tight_layout(); plt.show()

## 7. ALL neurons: which behavior matters more?

One neuron isn't enough to conclude anything. Let's measure, for **every one of the 8,372 neurons**, how strongly its response correlates with pupil size and with running speed. Then we'll look at the whole population.

The helper below is a fast way to compute the correlation for all neurons at once. You can treat it as a tool — the important part is what it *returns*: one correlation number per neuron.

In [ ]:
def correlation_with_every_neuron(behavior_variable, R):
    """Pearson correlation between one behavior variable and each neuron's response.
    Returns an array with one correlation value per neuron."""
    x = behavior_variable - behavior_variable.mean()
    Rc = R - R.mean(axis=0)
    numerator = (Rc * x[:, None]).sum(axis=0)
    denominator = np.sqrt((Rc**2).sum(axis=0) * (x**2).sum()) + 1e-12
    return numerator / denominator

corr_pupil   = correlation_with_every_neuron(pupil_size, R)
corr_running = correlation_with_every_neuron(running_speed, R)

print("PUPIL   -> mean corr %.3f | neurons with corr>0.1: %.1f%% | strongest %.2f" % (
      np.nanmean(corr_pupil),   100*np.mean(corr_pupil>0.1),   np.nanmax(corr_pupil)))
print("RUNNING -> mean corr %.3f | neurons with corr>0.1: %.1f%% | strongest %.2f" % (
      np.nanmean(corr_running), 100*np.mean(corr_running>0.1), np.nanmax(corr_running)))

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(corr_pupil,   bins=50, alpha=0.6, label="pupil size")
plt.hist(corr_running, bins=50, alpha=0.6, label="running speed")
plt.axvline(0, color="k", linewidth=0.8)
plt.xlabel("correlation between behavior and neuron's response")
plt.ylabel("number of neurons")
plt.title("How each behavior relates to the 8,372 neurons")
plt.legend(); plt.show()

## 8. What we found (write this down — it's a real result!)

On this mouse:
- **Pupil size (arousal) is clearly related to neural activity** — the pupil histogram sits shifted to the *right* of zero, and about a third of neurons have a meaningful positive correlation.
- **Running speed barely relates to activity here** — its histogram is centered near zero. Why? Because **this mouse hardly ever ran** (Section 5). No running, no running effect to detect.

**This is genuinely interesting.** The textbook says locomotion boosts visual cortex — but that only shows up if the animal moves. In *this* animal, *arousal* (pupil) is the behavior that carries the signal.

> **Why this matters for our project:** behavior clearly *does* relate to neural activity (at least pupil does). So when we build a prediction model in Weeks 3–4, adding behavior should help us predict neurons better — especially the arousal-sensitive ones. We now have a concrete prediction to test.

## 9. Save our work for next week

Reloading 6,000 files every time is slow. Let's save the two tables (and behavior) into a single compact file so Week 3 starts instantly.

In [ ]:
# Also load the train/validation/test labels so we can split properly next week
tiers = np.load(f"{FILENAME}/meta/trials/tiers.npy", allow_pickle=True)

np.savez_compressed("sensorium_processed.npz",
                    responses=R, behavior=B, tiers=tiers)
print("Saved sensorium_processed.npz")

# Optional: download it to your computer so you keep it (Colab sessions reset).
# Uncomment the next two lines to download:
# from google.colab import files
# files.download('sensorium_processed.npz')

## Exercises

1. In Section 6, change `neuron = 0` to a few other numbers (e.g. 100, 5000). Do some neurons care about pupil more than others?
2. Find the **most pupil-correlated neuron** and plot it. Hint: `best = np.argmax(corr_pupil)` gives its index; then reuse the Section-6 scatter code with `neuron = best`.
3. In Section 7, how many neurons have a *negative* pupil correlation (fire *less* when aroused)? Hint: `np.mean(corr_pupil < 0)`.

_Your notes:_



## Recap

- You loaded **real neural recordings** — 8,372 neurons, ~6,000 trials.
- You learned to **look at your data first** (and found the mouse rarely ran).
- You measured how **pupil** and **running** relate to every neuron, and found **arousal carries the signal in this mouse**.
- You saved processed data for next week.

### Next week (Week 3)
We build our first real **prediction model**: given information about a trial, can we predict how a neuron will respond? We'll start the *stimulus-only* baseline — the model we'll later add behavior to.

Fantastic work — you're doing real neuroscience now. 🧠🐭